# Section 5: Metadata Missingness and Usability Audit
Examines the demographic data targeting splits across the operational dataset.

In [1]:
import pandas as pd
import os
from IPython.display import display

OUTPUT_ROOT = r"C:\SKIN CANCER v2\pipe output"
d_audit = os.path.join(OUTPUT_ROOT, "audit_reports")
os.makedirs(d_audit, exist_ok=True)

## Load Restricted Operational Dataset

In [2]:
train_path = os.path.join(OUTPUT_ROOT, "manifests", "training_eligible_manifest.csv")
df_eval    = pd.read_csv(train_path)
print(f"Loaded training eligible dataset: {len(df_eval):,} rows.")
print(f"Class counts:")
print(df_eval["final_authoritative_label"].value_counts().to_string())

Loaded training eligible dataset: 20,664 rows.
Class counts:
final_authoritative_label
NV     12851
MEL     4504
BCC     3309


## Audit: Overall Database Missingness

In [3]:
fields    = ["age_approx", "sex", "anatom_site_general", "lesion_id"]
total_rows = len(df_eval)

overall_missing = []
for f in fields:
    n_present = int(df_eval[f].notna().sum()) if f in df_eval.columns else 0
    m_missing = total_rows - n_present
    p_missing = round((m_missing / total_rows) * 100, 2)
    overall_missing.append({
        "field_name": f, "total_rows": total_rows,
        "non_missing_count": n_present, "missing_count": m_missing,
        "missing_pct": p_missing
    })

df_overall = pd.DataFrame(overall_missing)
df_overall.to_csv(os.path.join(d_audit, "metadata_missingness_overall.csv"), index=False)
print("=== OVERALL METADATA MISSINGNESS ===")
display(df_overall)

=== OVERALL METADATA MISSINGNESS ===


,field_name,total_rows,non_missing_count,missing_count,missing_pct
0,age_approx,20664,20266,398,1.93
1,sex,20664,20316,348,1.68
2,anatom_site_general,20664,18386,2278,11.02
3,lesion_id,20664,18779,1885,9.12


## Audit: Missingness Sliced by Disease Class

In [4]:
class_missingness = []
for c in ["NV", "MEL", "BCC"]:
    df_c    = df_eval[df_eval["final_authoritative_label"] == c]
    c_total = len(df_c)
    for f in fields:
        n_present = int(df_c[f].notna().sum()) if f in df_c.columns else 0
        m_missing = c_total - n_present
        p_missing = round((m_missing / c_total) * 100, 2) if c_total > 0 else 0
        class_missingness.append({
            "field_name": f, "class_label": c, "class_total": c_total,
            "non_missing_count": n_present, "missing_count": m_missing, "missing_pct": p_missing
        })

df_class_miss = pd.DataFrame(class_missingness)
df_class_miss.to_csv(os.path.join(d_audit, "metadata_missingness_by_class.csv"), index=False)
print("=== METADATA MISSINGNESS BY CLASS ===")
display(df_class_miss)

=== METADATA MISSINGNESS BY CLASS ===


,field_name,class_label,class_total,non_missing_count,missing_count,missing_pct
0,age_approx,NV,12851,12544,307,2.39
1,sex,NV,12851,12590,261,2.03
2,anatom_site_general,NV,12851,10768,2083,16.21
3,lesion_id,NV,12851,11303,1548,12.05
4,age_approx,MEL,4504,4419,85,1.89
5,sex,MEL,4504,4423,81,1.80
6,anatom_site_general,MEL,4504,4377,127,2.82
7,lesion_id,MEL,4504,4167,337,7.48
8,age_approx,BCC,3309,3303,6,0.18
9,sex,BCC,3309,3303,6,0.18


## Missingness Overlap Analysis

In [5]:
metadata_fields = ["age_approx", "sex", "anatom_site_general", "lesion_id"]
metadata_fields = [f for f in metadata_fields if f in df_eval.columns]

df_eval_overlap = df_eval.copy()
for col in metadata_fields:
    df_eval_overlap[f"{col}_missing"] = df_eval_overlap[col].isna()

df_eval_overlap["num_missing_metadata_fields"] = df_eval_overlap[
    [f"{c}_missing" for c in metadata_fields]
].sum(axis=1)

missing_count_by_disease = (
    df_eval_overlap.groupby(["final_authoritative_label", "num_missing_metadata_fields"])
    .size().reset_index(name="row_count")
    .sort_values(["final_authoritative_label", "num_missing_metadata_fields"])
)

def make_missing_pattern(row):
    missing_cols = [c for c in metadata_fields if pd.isna(row[c])]
    return "none_missing" if not missing_cols else "|".join(missing_cols)

df_eval_overlap["missing_pattern"] = df_eval_overlap.apply(make_missing_pattern, axis=1)

missing_pattern_by_disease = (
    df_eval_overlap.groupby(["final_authoritative_label", "missing_pattern"])
    .size().reset_index(name="row_count")
    .sort_values(["final_authoritative_label", "row_count"], ascending=[True, False])
)

missing_count_by_disease.to_csv(os.path.join(d_audit, "metadata_missing_field_count_by_disease.csv"), index=False)
missing_pattern_by_disease.to_csv(os.path.join(d_audit, "metadata_missing_overlap_patterns_by_disease.csv"), index=False)
print("=== MISSING FIELD COUNT BY DISEASE ===")
display(missing_count_by_disease)

=== MISSING FIELD COUNT BY DISEASE ===


,final_authoritative_label,num_missing_metadata_fields,row_count
0,BCC,0,3235
1,BCC,1,68
2,BCC,2,6
3,MEL,0,4083
4,MEL,1,316
5,MEL,2,37
6,MEL,3,32
7,MEL,4,36
8,NV,0,9733
9,NV,1,2387


## All-Metadata-Missing Drop Policy

In [6]:
all_metadata_missing_mask = df_eval[metadata_fields].isna().all(axis=1)
all_metadata_missing_rows = df_eval[all_metadata_missing_mask].copy()

print(f"Rows with all {len(metadata_fields)} metadata fields missing: {len(all_metadata_missing_rows):,}")
drop_counts_by_class = (
    all_metadata_missing_rows["final_authoritative_label"]
    .value_counts().rename_axis("final_authoritative_label").reset_index(name="drop_count")
)
print("=== DROP COUNTS BY DISEASE ===")
display(drop_counts_by_class)

drop_path = os.path.join(d_audit, "all_metadata_missing_rows_dropped.csv")
all_metadata_missing_rows.to_csv(drop_path, index=False)

df_eval_post_metadata = df_eval[~all_metadata_missing_mask].copy()
post_metadata_manifest_path = os.path.join(
    OUTPUT_ROOT, "manifests", "training_eligible_manifest_post_metadata.csv"
)
df_eval_post_metadata.to_csv(post_metadata_manifest_path, index=False)

print(f"\nDropped {len(all_metadata_missing_rows):,} rows with all metadata missing.")
print(f"Original rows : {len(df_eval):,}")
print(f"Post-metadata : {len(df_eval_post_metadata):,}")
print(f"Saved to      : {post_metadata_manifest_path}")

Rows with all 4 metadata fields missing: 151
=== DROP COUNTS BY DISEASE ===


,final_authoritative_label,drop_count
0,NV,115
1,MEL,36



Dropped 151 rows with all metadata missing.
Original rows : 20,664
Post-metadata : 20,513
Saved to      : C:\SKIN CANCER v2\pipe output\manifests\training_eligible_manifest_post_metadata.csv


## Lesion ID Split Support Verification

In [7]:
lesion_series = df_eval["lesion_id"].dropna() if "lesion_id" in df_eval.columns else pd.Series([], dtype=str)
unique_lesions = lesion_series.nunique()
lesion_counts  = lesion_series.value_counts()
multiple_imm   = int((lesion_counts > 1).sum())
pct_lesions    = round((len(lesion_series) / total_rows) * 100, 2)

lesion_report = [{
    "total_rows": total_rows,
    "rows_with_lesion_id": len(lesion_series),
    "rows_without_lesion_id": total_rows - len(lesion_series),
    "pct_rows_with_lesion_id": pct_lesions,
    "unique_lesion_ids": unique_lesions,
    "lesion_ids_with_multiple_images": multiple_imm
}]
df_lesion = pd.DataFrame(lesion_report)
df_lesion.to_csv(os.path.join(d_audit, "lesion_id_split_support_report.csv"), index=False)
display(df_lesion)

lesion_usable_for_split = pct_lesions >= 80 and multiple_imm > 0
print(f"\nlesion_id usable for split support: {lesion_usable_for_split}")

,total_rows,rows_with_lesion_id,rows_without_lesion_id,pct_rows_with_lesion_id,unique_lesion_ids,lesion_ids_with_multiple_images
0,20664,18779,1885,90.88,9735,3942



lesion_id usable for split support: True


## Generate Usage Policy

In [8]:
usage_policy = []
for f in fields:
    if f == "lesion_id":
        usability_class = "usable_for_split_support" if lesion_usable_for_split else "limited_use_only"
    else:
        f_pct = df_overall.loc[df_overall["field_name"] == f, "missing_pct"].values[0]
        if f_pct < 20:
            usability_class = "usable_for_audit"
        elif f_pct < 40:
            usability_class = "limited_use_only"
        else:
            usability_class = "not_usable"
    usage_policy.append({
        "field_name":            f,
        "usability_class":       usability_class,
        "use_for_audit":         usability_class != "not_usable",
        "use_for_reporting":     usability_class != "not_usable",
        "use_for_split_support": usability_class == "usable_for_split_support",
        "use_for_bias_analysis": f in ["age_approx","sex","anatom_site_general"] and usability_class in ["usable_for_audit","usable_for_split_support"],
        "use_for_model_input_v1": False,
        "notes": "Derived from observed missingness."
    })
df_policy = pd.DataFrame(usage_policy)
df_policy.to_csv(os.path.join(d_audit, "metadata_usage_policy.csv"), index=False)
print("=== METADATA USAGE POLICY ===")
display(df_policy)

=== METADATA USAGE POLICY ===


,field_name,usability_class,use_for_audit,use_for_reporting,use_for_split_support,use_for_bias_analysis,use_for_model_input_v1,notes
0,age_approx,usable_for_audit,True,True,False,True,False,Derived from observed missingness.
1,sex,usable_for_audit,True,True,False,True,False,Derived from observed missingness.
2,anatom_site_general,usable_for_audit,True,True,False,True,False,Derived from observed missingness.
3,lesion_id,usable_for_split_support,True,True,True,False,False,Derived from observed missingness.


In [9]:
# Final summary
print("=" * 60)
print("  05_metadata_audit -- FINAL SUMMARY")
print("=" * 60)
print(f"\nTotal rows loaded               : {total_rows:,}")
print(f"Metadata matched rows           : {int(df_eval['metadata_row_found'].sum()) if 'metadata_row_found' in df_eval.columns else 'n/a'}")
for f in fields:
    miss = int(df_eval[f].isna().sum()) if f in df_eval.columns else total_rows
    print(f"  Missing {f:<26}: {miss:,}")
print(f"\nlesion_id usable for split support: {lesion_usable_for_split}")
print(f"\nRows with all metadata missing (dropped): {len(all_metadata_missing_rows):,}")
print(f"Post-metadata manifest rows              : {len(df_eval_post_metadata):,}")
print(f"\nOutput file verification:")
_files = [
    os.path.join(d_audit, "metadata_missingness_overall.csv"),
    os.path.join(d_audit, "metadata_missingness_by_class.csv"),
    os.path.join(d_audit, "metadata_usage_policy.csv"),
    os.path.join(d_audit, "all_metadata_missing_rows_dropped.csv"),
    post_metadata_manifest_path,
]
for p in _files:
    exists = os.path.exists(p)
    size   = os.path.getsize(p) if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {os.path.basename(p):<45} {size:>10,} bytes")
print("=" * 60)

  05_metadata_audit -- FINAL SUMMARY

Total rows loaded               : 20,664
Metadata matched rows           : 20664
  Missing age_approx                : 398
  Missing sex                       : 348
  Missing anatom_site_general       : 2,278
  Missing lesion_id                 : 1,885

lesion_id usable for split support: True

Rows with all metadata missing (dropped): 151
Post-metadata manifest rows              : 20,513

Output file verification:
  [OK] metadata_missingness_overall.csv                     203 bytes
  [OK] metadata_missingness_by_class.csv                    500 bytes
  [OK] metadata_usage_policy.csv                            506 bytes
  [OK] all_metadata_missing_rows_dropped.csv             53,140 bytes
  [OK] training_eligible_manifest_post_metadata.csv   6,852,425 bytes
